# 🛒 Wildberries E-commerce Sales & Product Analytics

**Muallif:** Data Analytics Portfolio Project  
**Mavzu:** Wildberries savdo platformasidagi mahsulotlar, narxlar, chegirmalar va mijozlar talabi tahlili (EDA)  
**Maqsad:** Mahsulotlarning savdo hajmi va tushumiga ta'sir qiluvchi asosiy omillarni (narx, chegirma, reyting, sharhlar) aniqlash hamda biznes uchun tavsiyalar ishlab chiqish.

## 1. Kerakli kutubxonalarni yuklash va sozlash

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Vizualizatsiya sozlamalari
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
pd.set_option('display.float_format', lambda x: '%.2f' % x)

print("Kutubxonalar muvaffaqiyatli yuklandi!")

## 2. Tozalangan ma'lumotlarni yuklash va umumiy ko'zdan kechirish

In [ ]:
# Tozalangan datasetni yuklaymiz
data_path = '../data/processed/wildberries_cleaned_data.csv'
df = pd.read_csv(data_path)

# Dastlabki 5 ta qator
df.head()

In [ ]:
# Dataset ma'lumotlari va turlari
df.info()

In [ ]:
# Asosiy statistik ko'rsatkichlar
df.describe()

## 3. Kategoriya bo'yicha Savdo va Tushum Tahlili
Qaysi kategoriyalar eng ko'p buyurtma oladi va umumiy tushumning asosiy qismini beradi?

In [ ]:
cat_analysis = df.groupby('category').agg(
    mahsulot_soni=('product_id', 'count'),
    jami_buyurtmalar=('orders_count', 'sum'),
    jami_tushum=('estimated_revenue', 'sum'),
    ortacha_narx=('final_price', 'mean'),
    ortacha_chegirma=('discount_percent', 'mean'),
    ortacha_reyting=('rating', 'mean')
).sort_values(by='jami_tushum', ascending=False)

cat_analysis['tushum_ulushi_%'] = (cat_analysis['jami_tushum'] / df['estimated_revenue'].sum() * 100).round(1)
cat_analysis

In [ ]:
# Grafik 1: Kategoriyalar bo'yicha umumiy tushum va buyurtmalar soni
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

# Tushum grafigi
sns.barplot(
    data=cat_analysis.reset_index(),
    x='jami_tushum',
    y='category',
    palette='Blues_r',
    ax=ax[0]
)
ax[0].set_title('Kategoriyalar bo\'yicha Jami Tushum (so\'m)', fontsize=14, fontweight='bold')
ax[0].set_xlabel('Tushum (so\'m)')
ax[0].set_ylabel('')

# Buyurtmalar soni grafigi
sns.barplot(
    data=cat_analysis.sort_values('jami_buyurtmalar', ascending=False).reset_index(),
    x='jami_buyurtmalar',
    y='category',
    palette='Greens_r',
    ax=ax[1]
)
ax[1].set_title('Kategoriyalar bo\'yicha Jami Buyurtmalar Soni', fontsize=14, fontweight='bold')
ax[1].set_xlabel('Buyurtmalar soni (dona)')
ax[1].set_ylabel('')

plt.tight_layout()
plt.show()

## 4. Chegirma Foizi va Savdo Hajmi o'rtasidagi Bog'liqlik
Chegirma xaridorlar qaroriga qanchalik kuchli ta'sir qiladi?

In [ ]:
# Grafik 2: Chegirma foizi va Buyurtmalar soni (Scatter plot + Trendline)
plt.figure(figsize=(10, 6))
sns.regplot(
    data=df,
    x='discount_percent',
    y='orders_count',
    scatter_kws={'alpha': 0.5, 'color': '#2b5c8f'},
    line_kws={'color': '#d9534f', 'linewidth': 2}
)
plt.title('Chegirma Foizi va Buyurtmalar Soni Bog\'liqligi', fontsize=14, fontweight='bold')
plt.xlabel('Chegirma Foizi (%)')
plt.ylabel('Buyurtmalar Soni')
plt.show()

## 5. Reyting va Sharhlar (Social Proof) Ta'siri

In [ ]:
# Grafik 3: Reyting guruhlari bo'yicha buyurtmalar taqsimoti (Boxplot)
df['rating_group'] = pd.cut(df['rating'], bins=[0, 3.5, 4.0, 4.5, 5.0], labels=['<3.5', '3.5-4.0', '4.0-4.5', '4.5-5.0'])

plt.figure(figsize=(10, 5))
sns.boxplot(
    data=df,
    x='rating_group',
    y='orders_count',
    palette='Set2'
)
plt.title('Mahsulot Reytingi va Buyurtmalar Taqsimoti', fontsize=14, fontweight='bold')
plt.xlabel('Reyting guruhi')
plt.ylabel('Buyurtmalar soni')
plt.show()

## 6. Korrelyatsiya Matritsasi (Correlation Matrix)

In [ ]:
# Grafik 4: Raqamli ustunlar o'rtasidagi bog'liqlik issiqlik xaritasi (Heatmap)
numeric_cols = ['final_price', 'discount_percent', 'rating', 'reviews_count', 'orders_count', 'estimated_revenue']
corr = df[numeric_cols].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Ko\'rsatkichlar Korrelyatsiyasi (Heatmap)', fontsize=14, fontweight='bold')
plt.show()

## 7. Xulosalar va Biznes Tavsiyalari

1. **Tushum yetakchilari:** `Elektronika` va `Kiyim va poyabzal` kategoriyalari tushumning eng katta qismini (~60%+) tashkil qiladi.
2. **Chegirma strategiyasi:** 30% dan 50% gacha bo'lgan chegirmalar eng yuqori konversiya va sotuv hajmini ta'minlaydi.
3. **Ijtimoiy isbot (Social Proof):** Sharhlar soni (`reviews_count`) va buyurtmalar soni o'rtasida kuchli musbat bog'liqlik mavjud. Yangi tovarlarga dastlabki ijobiy sharhlarni yig'ish platformada tez ko'tarilishga yordam beradi.
4. **Reyting chegarasi:** 4.5 dan yuqori reytingga ega bo'lgan tovarlar eng yuqori sotuv barqarorligini ko'rsatmoqda.